In [1]:
import sys

sys.path.append("..")
import os
import time
import glob
import numpy as np
import pandas as pd
from typing import List, Dict, Union, Tuple
from utils.data import load_yaml
from imaris.imaris import ImarisDataObject
from parsers.spot_track_parser import SpotTrackParserDistributed
from parsers.spot_track_object_parser import SpotTrackObjectParserDistributed
from tqdm.notebook import tqdm

import csv

Get Stats From Parser

In [2]:
# test ims file path
test_ims_file = "../../data/imaris_parser_test_files/Spots/KO Sec1 Roi1 3x3 80min.ims"

In [3]:
parser = SpotTrackParserDistributed(test_ims_file)
out = parser.inspect(1)
parser_df = out["final_df"]
parser_stat_names = list(parser_df.columns)

Get Stats From Test Files

In [4]:
# track stats folder
track_stats_dir = "../../data/imaris_parser_test_files/Spots/KO Sec1 Roi1 3x3 80min-testtracks_Statistics"

# object stats folder
# object_stats_dir = (
#     "../../data/imaris_parser_test_files/Spots/KO Sec1 Roi1 3x3 80min_Statistics"
# )

In [5]:
track_stats_files = glob.glob(os.path.join(track_stats_dir, "*.csv"))

In [6]:
def get_stats_df(path: str) -> Tuple[str, pd.DataFrame]:
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.reader(f)
        dataframe = []
        for row in reader:
            dataframe.append(row)

    # get the stats name
    stat_name = dataframe[1][0]

    # the first three rows from imaris export needs to be dropped
    dataframe = pd.DataFrame(dataframe[4:], columns=dataframe[3])
    # out = {stat_name: dataframe}
    return (stat_name, dataframe)


index = 6
items = get_stats_df(track_stats_files[index])
print(items[0], "\n")
items[1]

Track Volume Mean 



,Track Volume Mean,Unit,Category,ID,
0,648.609,µm^3,Track,1000000039,
1,648.609,µm^3,Track,1000000042,
2,648.609,µm^3,Track,1000000060,
3,648.609,µm^3,Track,1000000065,
4,648.609,µm^3,Track,1000000073,
...,...,...,...,...,...
751,648.609,µm^3,Track,1000654755,
752,648.609,µm^3,Track,1000656003,
753,648.609,µm^3,Track,1000664852,
754,648.609,µm^3,Track,1000666870,


In [7]:
# get all the stat from the test files into a dict
test_stats_dict = {}
for stat_test_path in tqdm(track_stats_files):
    items = get_stats_df(stat_test_path)
    if items[0] in test_stats_dict.keys():
        raise ValueError
    else:
        test_stats_dict[items[0]] = items[1]

  0%|          | 0/128 [00:00<?, ?it/s]

In [127]:
print(f"Num stats to evaluate: {len(parser_stat_names)}")

for stat_name in tqdm(parser_stat_names):
    # get test stats df
    if stat_name != "Track_ID":

        test_df = test_stats_dict[stat_name]

        # it seems like if channel and image information is given the df will modify its name
        # we need to drop these
        stat_name_our_df = None
        if "Channel" in list(test_df.columns) and "Image" in list(test_df.columns):
            index = stat_name.find("Ch=") - 1
            stat_name_our_df = stat_name[:index]

        if "Spots" in list(test_df.columns):
            if stat_name_our_df:
                index = stat_name_our_df.find("Spots=") - 1
                stat_name_our_df = stat_name_our_df[:index]
            else:
                index = stat_name.find("Spots=") - 1
                stat_name_our_df = stat_name[:index]

        # get df to compare to
        if stat_name_our_df:
            test_df = test_stats_dict[stat_name].filter([stat_name_our_df, "ID"])
        else:
            test_df = test_stats_dict[stat_name].filter([stat_name, "ID"])

        # get the df we generate
        our_df = parser_df.filter([stat_name, "Track_ID"]).rename(
            columns={"Track_ID": "ID"}
        )
        our_df = our_df.reset_index().drop("index", axis=1)
        our_df["ID"] = our_df["ID"].astype(str)
        our_df[stat_name] = our_df[stat_name].astype(float).map(lambda x: f"{x:.3f}")
        our_df[stat_name] = our_df[stat_name].astype(str)

        # it seems like if channel and image information is given the df will modify its name
        # we need to drop these
        if stat_name_our_df:
            our_df = our_df.rename(columns={stat_name: stat_name_our_df})

        if not (test_df.equals(our_df)):
            print(stat_name)
            print(stat_name_our_df)
            break

Num stats to evaluate: 52


  0%|          | 0/52 [00:00<?, ?it/s]

In [122]:
our_df

,Track Intensity Center Mean,ID
0,18.069,1000000039
1,16.036,1000000042
2,10.000,1000000060
3,19.217,1000000065
4,14.783,1000000073
...,...,...
751,9.286,1000654755
752,13.900,1000656003
753,19.250,1000664852
754,51.182,1000666870


In [125]:
test_df

,ID
0,1000000039
1,1000000042
2,1000000060
3,1000000065
4,1000000073
...,...
751,1000654755
752,1000656003
753,1000664852
754,1000666870


In [92]:
t = test_stats_dict["Track Intensity Center Mean Ch=1 Img=1"]
o = parser_df.filter(["Track Intensity Sum Ch=3 Img=1", "Track_ID"]).rename(
    columns={"Track_ID": "ID"}
)
t

,Track Intensity Center Mean,Unit,Category,Channel,Image,ID,
0,18.069,,Track,1,Image 1,1000000039,
1,16.036,,Track,1,Image 1,1000000042,
2,10.000,,Track,1,Image 1,1000000060,
3,19.217,,Track,1,Image 1,1000000065,
4,14.783,,Track,1,Image 1,1000000073,
...,...,...,...,...,...,...,...
751,9.286,,Track,1,Image 1,1000654755,
752,13.900,,Track,1,Image 1,1000656003,
753,19.250,,Track,1,Image 1,1000664852,
754,51.182,,Track,1,Image 1,1000666870,


In [93]:
stat_name = "Track Intensity Center Mean"

In [ ]:
if "Channel" in list(t.columns) and "Image" in list(t.columns):
    stat_name_new = (
        f"{stat_name} Ch={t.iloc[0]['Channel']} Img={t.iloc[0]['Image'].split(" ")[-1]}"
    )
    t = t.rename(columns={stat_name: stat_name_new})

In [95]:
t

,Track Intensity Center Mean Ch=1 Img=1,Unit,Category,Channel,Image,ID,
0,18.069,,Track,1,Image 1,1000000039,
1,16.036,,Track,1,Image 1,1000000042,
2,10.000,,Track,1,Image 1,1000000060,
3,19.217,,Track,1,Image 1,1000000065,
4,14.783,,Track,1,Image 1,1000000073,
...,...,...,...,...,...,...,...
751,9.286,,Track,1,Image 1,1000654755,
752,13.900,,Track,1,Image 1,1000656003,
753,19.250,,Track,1,Image 1,1000664852,
754,51.182,,Track,1,Image 1,1000666870,


In [63]:
stat_name = f"{stat_name} Ch={t.iloc[0]}"

In [24]:
list(test_stats_dict.keys())

['Track Intensity Median Ch=3 Img=1',
 'Displacement Delta Length',
 'Track Intensity Max Ch=3 Img=1',
 'Speed',
 'Time Index',
 'Time',
 'Track Volume Mean',
 'Track Intensity Max Ch=1 Img=1',
 'Intensity Center Ch=3 Img=1',
 'Average Distance To 9 Nearest Neighbours',
 'Track Displacement Length',
 'Track Position Start',
 'Intensity Sum of Square Ch=2 Img=1',
 'Distance To Nearest Neighbour',
 'Intensity Sum Ch=2 Img=1',
 'Position Y',
 'Intensity Sum Ch=3 Img=1',
 'Track Intensity Median Ch=1 Img=1',
 'Track Speed Mean',
 'Displacement Length',
 'Distance to Image Border XY Img=1',
 'Intensity Min Ch=1 Img=1',
 'Displacement Delta',
 'Track Straightness',
 'Track Intensity Sum Ch=2 Img=1',
 'Intensity Sum Ch=1 Img=1',
 'Track Intensity Median Ch=2 Img=1',
 'Track Number of Fusions',
 'Position Z',
 'Track Intensity Sum Ch=3 Img=1',
 'Intensity Min Ch=2 Img=1',
 'Average Distance To 5 Nearest Neighbours',
 'Intensity Center Ch=1 Img=1',
 'Displacement X',
 'Track Position',
 'Track 

In [ ]:
test_stats_dict["Intensity Sum of Square Ch=2 Img=1"]

,Intensity Sum of Square,Unit,Category,Channel,Image,Time,TrackID,ID,
0,80128.000,,Spot,2,Image 1,1,1000000039,39,
1,64960.000,,Spot,2,Image 1,1,1000000042,42,
2,38545.000,,Spot,2,Image 1,1,1000000060,60,
3,56544.000,,Spot,2,Image 1,1,1000000065,65,
4,46521.000,,Spot,2,Image 1,1,1000000073,73,
...,...,...,...,...,...,...,...,...,...
38975,29738.000,,Spot,2,Image 1,55,1000666870,1294433,
38976,42695.000,,Spot,2,Image 1,59,1000666870,1294434,
38977,43121.000,,Spot,2,Image 1,61,1000666870,1294435,
38978,74319.000,,Spot,2,Image 1,51,1000682257,1294476,


In [31]:
list(parser_df.columns)

['Track Ar1 Mean',
 'Track Ar1 X',
 'Track Ar1 Y',
 'Track Ar1 Z',
 'Track Area Mean',
 'Track Diameter Mean',
 'Track Displacement Length',
 'Track Displacement X',
 'Track Displacement Y',
 'Track Displacement Z',
 'Track Duration',
 'Track Intensity Center Mean Ch=1 Img=1',
 'Track Intensity Center Mean Ch=2 Img=1',
 'Track Intensity Center Mean Ch=3 Img=1',
 'Track Intensity Max Ch=1 Img=1',
 'Track Intensity Max Ch=2 Img=1',
 'Track Intensity Max Ch=3 Img=1',
 'Track Intensity Mean Ch=1 Img=1',
 'Track Intensity Mean Ch=2 Img=1',
 'Track Intensity Mean Ch=3 Img=1',
 'Track Intensity Median Ch=1 Img=1',
 'Track Intensity Median Ch=2 Img=1',
 'Track Intensity Median Ch=3 Img=1',
 'Track Intensity Min Ch=1 Img=1',
 'Track Intensity Min Ch=2 Img=1',
 'Track Intensity Min Ch=3 Img=1',
 'Track Intensity StdDev Ch=1 Img=1',
 'Track Intensity StdDev Ch=2 Img=1',
 'Track Intensity StdDev Ch=3 Img=1',
 'Track Intensity Sum Ch=1 Img=1',
 'Track Intensity Sum Ch=2 Img=1',
 'Track Intensity Su

In [32]:
len(test_stats_dict.keys())

128

TypeError: unsupported operand type(s) for -: 'str' and 'list'